# Importing Libs

In [ ]:
# Cross-Platform Environment Setup
import os
import sys

# Detect environment
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

IN_KAGGLE = 'KAGGLE_URL_BASE' in os.environ

print(f"🌐 Environment — Colab: {IN_COLAB} | Kaggle: {IN_KAGGLE} | Local: {not IN_COLAB and not IN_KAGGLE}")

In [ ]:
!pip install -qU ultralytics supervision

In [ ]:
import os
import sys
import time

import cv2
import torch
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import supervision as sv
from IPython.display import display, clear_output
import ipywidgets as widgets

# ── Device ──────────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Using device: {device}")

# ── Output path (environment-aware) ─────────────────────────────────────────
if IN_KAGGLE:
    OUTPUT_DIR = Path("/kaggle/working")
elif IN_COLAB:
    OUTPUT_DIR = Path("/content")
else:
    OUTPUT_DIR = Path(".")   # current working directory

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VIDEO_PATH = str(OUTPUT_DIR / "output_annotated.mp4")
print(f"📂 Output will be saved to: {OUTPUT_VIDEO_PATH}")

# Load The Model

In [ ]:
# Configuration
MODEL_PATH = "/kaggle/input/models/ahmedhossamelrayes/acc-detect/pytorch/default/2/best (3).pt"
CONFIDENCE_THRESHOLD = 0.5

if not Path(MODEL_PATH).exists():
    print(f"⚠️  Model not found at {MODEL_PATH}. Checking standard location...")

model = YOLO(MODEL_PATH)
model.to(device)
print(f"✅ Model loaded successfully from: {MODEL_PATH}")

# Detection Helper Functions

In [ ]:
def detect_and_annotate(frame, model, confidence=0.5):
    """Run inference and return an annotated frame with detections and timing."""
    start_time = time.time()
    results = model.predict(frame, conf=confidence, device=device, verbose=False)
    inference_time = (time.time() - start_time) * 1000

    detections = sv.Detections.from_ultralytics(results[0])

    box_annotator   = sv.BoxAnnotator(thickness=2)
    label_annotator = sv.LabelAnnotator(text_scale=0.5, text_thickness=1)

    labels = [
        f"{model.names[class_id]} {conf:.2f}"
        for class_id, conf in zip(detections.class_id, detections.confidence)
    ] if len(detections) > 0 else []

    annotated_frame = box_annotator.annotate(scene=frame.copy(), detections=detections)
    annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)

    return annotated_frame, detections, inference_time


def add_stats_overlay(frame, fps, inference_time, detection_count):
    """Overlay a semi-transparent stats box in the top-left corner."""
    overlay = frame.copy()
    cv2.rectangle(overlay, (10, 10), (250, 100), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

    cv2.putText(frame, f"FPS: {fps:.1f}",                  (20, 35),  cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv2.putText(frame, f"Inference: {inference_time:.1f}ms", (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv2.putText(frame, f"Detections: {detection_count}",    (20, 85),  cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    return frame


def show_frame(window_name, frame):
    """Display a frame inline (Colab/Kaggle) or in a cv2 window (local)."""
    if IN_COLAB or IN_KAGGLE:
        _, encoded_img = cv2.imencode('.jpg', frame)
        display(widgets.Image(value=encoded_img.tobytes()))
        clear_output(wait=True)
    else:
        cv2.imshow(window_name, frame)


def build_video_writer(output_path, fps, width, height):
    """Create a cv2.VideoWriter with the mp4v codec."""
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    if not writer.isOpened():
        raise RuntimeError(f"❌ Could not open VideoWriter for path: {output_path}")
    return writer


def download_output_video(path):
    """
    Trigger a file download after processing is complete.
      - Colab  : uses google.colab.files.download()
      - Kaggle : instructs the user to use the Output tab
      - Local  : prints the absolute file path
    """
    abs_path = str(Path(path).resolve())
    file_size_mb = Path(path).stat().st_size / (1024 ** 2)
    print(f"\n💾 Output video saved — {file_size_mb:.1f} MB → {abs_path}")

    if IN_COLAB:
        from google.colab import files
        print("📥 Starting download to your machine...")
        files.download(path)
    elif IN_KAGGLE:
        print("📁 Kaggle: Open the ▶ Output tab on the right panel to download the file.")
    else:
        print(f"📁 Local: Video is ready at  {abs_path}")

# Test with Video File 🎬

In [ ]:
VIDEO_PATH = "/kaggle/input/datasets/ahmedhossamelrayes/ocr-trying/Car_Accident_Video_In_a_dramatic_style_two_cars_a_silver_hatchback_kX_6e5FI.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print(f"❌ Error: Could not open video file: {VIDEO_PATH}")
else:
    # ── Read source video properties ─────────────────────────────────────────
    src_width    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    src_height   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    src_fps      = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"✅ Video loaded : {VIDEO_PATH}")
    print(f"   Resolution   : {src_width}×{src_height}")
    print(f"   FPS          : {src_fps:.2f}")
    print(f"   Total frames : {total_frames}")
    print(f"   Output path  : {OUTPUT_VIDEO_PATH}")

    # ── Initialize VideoWriter ────────────────────────────────────────────────
    writer = build_video_writer(OUTPUT_VIDEO_PATH, src_fps, src_width, src_height)
    print("\n🎬 Processing video...\n")

    frame_count   = 0
    fps_start     = time.time()
    display_fps   = 0.0

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame_count += 1

            # Inference + annotation
            annotated_frame, detections, inference_time = detect_and_annotate(
                frame, model, CONFIDENCE_THRESHOLD
            )

            # Update display FPS every 10 frames
            if frame_count % 10 == 0:
                elapsed = time.time() - fps_start
                display_fps = 10 / elapsed if elapsed > 0 else 0.0
                fps_start = time.time()

                # Progress log (print overwrites the same line in most notebooks)
                pct = (frame_count / total_frames * 100) if total_frames > 0 else 0
                print(f"\r⏳ Frame {frame_count}/{total_frames} ({pct:.1f}%)  "
                      f"| Display FPS: {display_fps:.1f}  "
                      f"| Inference: {inference_time:.1f} ms", end="", flush=True)

            # Overlay stats
            annotated_frame = add_stats_overlay(
                annotated_frame, display_fps, inference_time, len(detections)
            )

            # Accident alert banner
            if len(detections) > 0:
                cv2.putText(
                    annotated_frame, "ACCIDENT DETECTED!",
                    (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2
                )

            # ── Write frame to output file ────────────────────────────────────
            writer.write(annotated_frame)

            # ── Display frame inline / in window ─────────────────────────────
            show_frame("Safespace Model Test", annotated_frame)

            # Local: allow 'q' to quit early
            if not (IN_COLAB or IN_KAGGLE):
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    print("\n⏹️  Stopped early by user.")
                    break

    finally:
        # ── Always release resources, even if an error occurs ─────────────────
        cap.release()
        writer.release()
        if not (IN_COLAB or IN_KAGGLE):
            cv2.destroyAllWindows()

    print(f"\n✅ Done! Processed {frame_count} frames.")

    # ── Save / download the output video ─────────────────────────────────────
    download_output_video(OUTPUT_VIDEO_PATH)